# GNN-Pruning — full pipeline (Colab / A100)

Reproduces **all** results: 5 methods (dense + magnitude + 3 Wanda) × all
datasets × 4 architectures (GCN / GAT / GraphSAGE / GPR-GNN) × 9 sparsities,
with **3-seed** error bars on the core cells and **minibatch (neighbour-sampled)
training** for the large graphs (Reddit, ogbn-arxiv, Yelp, ogbn-products,
Flickr) — which fixes full-batch under-training (1 epoch = 1 gradient step).

- Minibatch is **pure PyTorch** — no `pyg-lib`/`torch-sparse` install.
- The sweep is **idempotent**: any cells already committed to `main` (e.g. the
  multi-seed core run on the Mac) are skipped; the rest run here. A fresh run with
  no prior results reproduces everything from scratch.
- Eval + Wanda activation-collection stay full-batch on the sparse path, so the
  4 pruning methods are unchanged. `reddit/gat` was removed (infeasible).

**Before running:** `Runtime → A100 GPU` (Colab Pro), enable background execution.


## 1. Confirm GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "Set Runtime → A100 GPU"
print("GPU:", torch.cuda.get_device_name(0),
      "| VRAM(GB):", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1),
      "| torch:", torch.__version__)

## 2. Mount Drive (dataset cache only — results come back as a download)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DATA = '/content/drive/MyDrive/gnn-pruning/data'
os.makedirs(DRIVE_DATA, exist_ok=True)
print('dataset cache:', DRIVE_DATA)

## 3. Clone `main` + install (no pyg-lib needed)

In [ ]:
%cd /content
![ -d GNN-Pruning-Research ] || git clone --branch main https://github.com/Mike-Mans/GNN-Pruning-Research.git
%cd /content/GNN-Pruning-Research
!git fetch origin && git checkout main && git pull --ff-only

In [ ]:
# Cache datasets on Drive (download once); results stay in the repo dir.
import os, shutil
if not os.path.islink('data'):
    if os.path.exists('data'): shutil.rmtree('data')
    os.symlink(DRIVE_DATA, 'data')
!pip -q install torch_geometric ogb rdkit
!pip -q install -e .
print('install done')

## 4. Run the full sweep — no-pruning first
Deletes the stale full-batch large cells so they retrain with the minibatch path; everything already committed (the multi-seed core) is skipped. `ogbn-products` downloads headlessly. Expect a while on A100.

In [ ]:
import os, subprocess, sys
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# stale full-batch large cells → remove so they retrain with minibatch (core kept)
!rm -rf results/*/reddit results/*/ogbn-arxiv results/*/yelp results/*/ogbn-products results/*/flickr

CONFIGS = {
    'no-pruning':      'src/gnn_pruning/configs/no_pruning.yaml',
    'magnitude':       'src/gnn_pruning/configs/magnitude.yaml',
    'wanda-uniform':   'src/gnn_pruning/configs/wanda_uniform.yaml',
    'wanda-degree':    'src/gnn_pruning/configs/wanda_degree.yaml',
    'wanda-per-class': 'src/gnn_pruning/configs/wanda_per_class.yaml',
}
for method, cfg in CONFIGS.items():           # full sweep (no --datasets filter)
    print(f'\n===== {method} =====', flush=True)
    rc = subprocess.run([sys.executable, '-m', 'gnn_pruning.cli', 'sweep',
                         '--method', method, '--config', cfg],
                        env={**os.environ, 'PYTHONUNBUFFERED': '1'}).returncode
    print(f'{method} done (rc={rc})', flush=True)
print('\nSWEEP COMPLETE')

## 5. Did minibatch help? (dense baselines vs the old full-batch)

In [ ]:
import pandas as pd, glob, json
print('=== large-dataset dense baselines (minibatch-trained) ===')
for f in sorted(glob.glob('results/no-pruning/*/*/seed-0/split-0/metrics.json')):
    d = f.split('/')[2]
    if d in {'reddit','ogbn-arxiv','yelp','ogbn-products','flickr'}:
        m = json.load(open(f))
        print(f"  {d:14s}/{f.split('/')[3]:10s} {m['metric_value']:.3f}  (best epoch {m['epoch_of_best_val']})")
print('\n--- failures (expect only reddit/gat) ---')
!grep -h -A1 FAILED results/*/run.log | grep -i error | sort -u | head

## 6. Download all results (summaries + metrics + plots) to merge locally
Excludes the large `*.pt` checkpoints (regenerable, gitignored).

In [ ]:
import shutil, os, glob
os.makedirs('/content/export', exist_ok=True)
files = (glob.glob('results/**/summary.csv', recursive=True)
         + glob.glob('results/**/metrics.json', recursive=True)
         + glob.glob('results/**/run.log', recursive=True)
         + glob.glob('results/**/*.png', recursive=True))
for f in files:
    dst = '/content/export/' + f
    os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(f, dst)
shutil.make_archive('/content/gnn_results', 'zip', '/content/export')
print('files:', len(files))
from google.colab import files; files.download('/content/gnn_results.zip')

## Notes
- **Resuming after a disconnect:** re-run cells 1–4; finished cells are skipped.
- **Merge locally:** unzip into the repo, then rebuild summaries + the report:
  `python -m gnn_pruning.cli ...` is not needed — the summaries in the zip already
  reflect all cells on Drive; locally, regenerate `results_comprehensive.md`.
- reddit/gat stays infeasible (attention OOM); it's the only expected failure.
